# Solution A: Traditional ML for Authorship Verification

**Category A**: Unsupervised or traditional machine learning-based approach

**Task**: Authorship Verification (AV) — Given two text sequences, determine if both were written by the same author.

**Approach**: Stylometric feature engineering + Gradient Boosting classifier

This notebook demonstrates **inference mode**: given an input CSV file with text pairs, it loads the pre-trained model and generates predictions.

## 1. Install Required Packages

In [ ]:
!pip install scikit-learn pandas numpy --quiet

## 2. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import re
import zlib
from collections import Counter
from sklearn.preprocessing import normalize

print("Imports successful.")

## 3. Feature Extraction Functions

These functions extract stylometric (writing style) features from text pairs.
The features capture unconscious writing habits such as:

- **Punctuation usage** (comma, period, exclamation rates)
- **Vocabulary richness** (TTR, hapax legomena, Yule's K)
- **Function word frequencies** (the, a, is, but, etc.)
- **Text structure** (sentence length, word length distributions)
- **Stylometric distances** (NCD, Burrows' Delta, Keselj dissimilarity)
- **TF-IDF cosine similarity** (character and word n-gram profiles)

In [ ]:
# ===== Stylometric Feature Functions =====

def vocabulary_richness(text):
    """Compute vocabulary richness: TTR, hapax ratio, avg word length."""
    words = text.lower().split()
    if not words:
        return 0.0, 0.0, 0.0
    total_words = len(words)
    unique_words = set(words)
    ttr = len(unique_words) / total_words
    word_counts = Counter(words)
    hapax = sum(1 for c in word_counts.values() if c == 1)
    hapax_ratio = hapax / total_words
    avg_word_len = np.mean([len(w) for w in words])
    return ttr, hapax_ratio, avg_word_len


def yules_k(words):
    """Yule's K measure of vocabulary richness."""
    if not words:
        return 0.0
    fd = Counter(words)
    N = len(words)
    M = sum(v * v for v in fd.values())
    return 10000 * (M - N) / (N * N) if N > 1 else 0.0


def punctuation_features(text):
    """Extract punctuation usage rates."""
    total_chars = len(text) if text else 1
    return {
        'comma_rate': text.count(',') / total_chars,
        'period_rate': text.count('.') / total_chars,
        'excl_rate': text.count('!') / total_chars,
        'quest_rate': text.count('?') / total_chars,
        'semicolon_rate': text.count(';') / total_chars,
        'colon_rate': text.count(':') / total_chars,
        'dash_rate': text.count('-') / total_chars,
        'quote_rate': (text.count('"') + text.count("'")) / total_chars,
        'paren_rate': (text.count('(') + text.count(')')) / total_chars,
        'ellipsis_rate': text.count('...') / total_chars,
    }


def function_word_features(text):
    """Extract function word frequency features."""
    function_words = [
        'the', 'a', 'an', 'and', 'or', 'but', 'if', 'in', 'on', 'at',
        'to', 'for', 'of', 'with', 'by', 'from', 'is', 'was', 'are', 'were',
        'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did',
        'will', 'would', 'could', 'should', 'may', 'might', 'can',
        'not', 'no', 'so', 'yet', 'just', 'also', 'now', 'then',
        'all', 'any', 'this', 'that', 'these', 'those',
        'i', 'me', 'my', 'we', 'us', 'our', 'you', 'your', 'he', 'him',
        'his', 'she', 'her', 'it', 'its', 'they', 'them', 'their',
        'about', 'after', 'before', 'into', 'through', 'up', 'down',
        'out', 'off', 'over', 'because', 'while', 'since',
    ]
    words = text.lower().split()
    total = len(words) if words else 1
    return {f'fw_{fw}': words.count(fw) / total for fw in function_words}


def text_structure_features(text):
    """Extract structural features about text composition."""
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    words = text.split()
    total_words = len(words) if words else 1
    total_chars = len(text) if text else 1
    sent_lengths = [len(s.split()) for s in sentences] if sentences else [0]
    word_lengths = [len(w) for w in words] if words else [0]
    return {
        'avg_sent_len': np.mean(sent_lengths),
        'std_sent_len': np.std(sent_lengths) if len(sent_lengths) > 1 else 0,
        'num_sentences': len(sentences),
        'avg_word_len': np.mean(word_lengths),
        'std_word_len': np.std(word_lengths) if len(word_lengths) > 1 else 0,
        'short_word_ratio': sum(1 for wl in word_lengths if wl <= 3) / total_words,
        'medium_word_ratio': sum(1 for wl in word_lengths if 4 <= wl <= 6) / total_words,
        'long_word_ratio': sum(1 for wl in word_lengths if wl >= 7) / total_words,
        'upper_ratio': sum(1 for c in text if c.isupper()) / total_chars,
        'digit_ratio': sum(1 for c in text if c.isdigit()) / total_chars,
        'space_ratio': text.count(' ') / total_chars,
        'total_words': total_words,
        'total_chars': total_chars,
    }


def extract_single_text_features(text):
    """Extract all stylometric features for a single text."""
    features = {}
    ttr, hapax_ratio, avg_wl = vocabulary_richness(text)
    features['ttr'] = ttr
    features['hapax_ratio'] = hapax_ratio
    features['vocab_avg_word_len'] = avg_wl
    features.update(punctuation_features(text))
    features.update(function_word_features(text))
    features.update(text_structure_features(text))
    return features


# ===== Stylometric Distance Measures =====

def normalised_compression_distance(text1, text2):
    """NCD from Cilibrasi & Vitanyi (2005). Lower = more similar."""
    b1, b2 = text1.encode('utf-8'), text2.encode('utf-8')
    c1, c2, c12 = len(zlib.compress(b1)), len(zlib.compress(b2)), len(zlib.compress(b1 + b2))
    max_c = max(c1, c2)
    return (c12 - min(c1, c2)) / max_c if max_c > 0 else 0.0


def burrows_delta(text1, text2):
    """Simplified Burrows' Delta. Lower = more similar style."""
    w1, w2 = text1.lower().split(), text2.lower().split()
    if not w1 or not w2:
        return 0.0
    f1 = {w: c / len(w1) for w, c in Counter(w1).items()}
    f2 = {w: c / len(w2) for w, c in Counter(w2).items()}
    all_w = set(f1) | set(f2)
    return sum(abs(f1.get(w, 0) - f2.get(w, 0)) for w in all_w) / len(all_w) if all_w else 0.0


def keselj_dissimilarity(text1, text2, n=3, L=500):
    """Keselj et al. (2003) char n-gram profile dissimilarity."""
    t1l, t2l = text1.lower(), text2.lower()
    ng1 = Counter(t1l[i:i+n] for i in range(len(t1l) - n + 1))
    ng2 = Counter(t2l[i:i+n] for i in range(len(t2l) - n + 1))
    tot1, tot2 = sum(ng1.values()) or 1, sum(ng2.values()) or 1
    p1 = {ng: c / tot1 for ng, c in ng1.most_common(L)}
    p2 = {ng: c / tot2 for ng, c in ng2.most_common(L)}
    all_ng = set(p1) | set(p2)
    if not all_ng:
        return 0.0
    return sum(((p1.get(ng, 0) - p2.get(ng, 0)) / ((p1.get(ng, 0) + p2.get(ng, 0)) / 2)) ** 2
               for ng in all_ng if (p1.get(ng, 0) + p2.get(ng, 0)) > 0)


# ===== Constants =====
STOPWORDS = frozenset({
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'is', 'was', 'are', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
    'should', 'i', 'me', 'my', 'we', 'us', 'you', 'your', 'he', 'him',
    'his', 'she', 'her', 'it', 'its', 'they', 'them', 'their', 'this',
    'that', 'not', 'no'
})
CONTRACTIONS = ["n't", "'s", "'re", "'ve", "'ll", "'d", "'m"]


# ===== Pairwise Feature Extraction =====

def extract_pair_features(text1, text2):
    """Extract pairwise difference and similarity features."""
    feats1 = extract_single_text_features(text1)
    feats2 = extract_single_text_features(text2)
    pair_features = {}
    for key in feats1:
        pair_features[f'diff_{key}'] = abs(feats1[key] - feats2[key])
    # Word Jaccard
    w1s, w2s = set(text1.lower().split()), set(text2.lower().split())
    union = w1s | w2s
    pair_features['word_jaccard'] = len(w1s & w2s) / len(union) if union else 0.0
    # Word bigram Jaccard
    w1, w2 = text1.lower().split(), text2.lower().split()
    bg1 = set(zip(w1[:-1], w1[1:])) if len(w1) > 1 else set()
    bg2 = set(zip(w2[:-1], w2[1:])) if len(w2) > 1 else set()
    pair_features['word_bigram_jaccard'] = len(bg1 & bg2) / len(bg1 | bg2) if (bg1 | bg2) else 0.0
    # Length ratios
    pair_features['char_len_ratio'] = min(len(text1), len(text2)) / max(len(text1), len(text2)) if max(len(text1), len(text2)) > 0 else 1.0
    wc1, wc2 = len(w1), len(w2)
    pair_features['word_count_ratio'] = min(wc1, wc2) / max(wc1, wc2) if max(wc1, wc2) > 0 else 1.0
    # Stylometric distances
    pair_features['ncd'] = normalised_compression_distance(text1, text2)
    pair_features['burrows_delta'] = burrows_delta(text1, text2)
    pair_features['keselj_3gram'] = keselj_dissimilarity(text1, text2, n=3, L=500)
    pair_features['keselj_4gram'] = keselj_dissimilarity(text1, text2, n=4, L=500)
    # Additional stylometric features
    pair_features['diff_yules_k'] = abs(yules_k(w1) - yules_k(w2))
    c1 = sum(text1.lower().count(c) for c in CONTRACTIONS) / (wc1 or 1)
    c2 = sum(text2.lower().count(c) for c in CONTRACTIONS) / (wc2 or 1)
    pair_features['diff_contraction_rate'] = abs(c1 - c2)
    sr1 = sum(1 for w in w1 if w in STOPWORDS) / (wc1 or 1)
    sr2 = sum(1 for w in w2 if w in STOPWORDS) / (wc2 or 1)
    pair_features['diff_stopword_ratio'] = abs(sr1 - sr2)
    sents1 = [s for s in re.split(r'[.!?]+', text1) if s.strip()]
    sents2 = [s for s in re.split(r'[.!?]+', text2) if s.strip()]
    sl1 = [len(s.split()) for s in sents1] if sents1 else [0]
    sl2 = [len(s.split()) for s in sents2] if sents2 else [0]
    pair_features['diff_sent_len_var'] = abs(np.var(sl1) - np.var(sl2))
    return pair_features


def build_feature_matrix(texts1, texts2, char_vectorizer, word_vectorizer):
    """Build the complete feature matrix for text pairs."""
    n = len(texts1)
    all_features = []
    for i in range(n):
        if (i + 1) % 2000 == 0:
            print(f"  Processing pair {i+1}/{n}...")
        feats = extract_pair_features(texts1[i], texts2[i])
        all_features.append(feats)
    feature_df = pd.DataFrame(all_features)
    # TF-IDF cosine similarities (efficient sparse computation)
    v1c = normalize(char_vectorizer.transform(texts1), norm='l2')
    v2c = normalize(char_vectorizer.transform(texts2), norm='l2')
    feature_df['tfidf_char_cosine'] = np.array(v1c.multiply(v2c).sum(axis=1)).flatten()
    v1w = normalize(word_vectorizer.transform(texts1), norm='l2')
    v2w = normalize(word_vectorizer.transform(texts2), norm='l2')
    feature_df['tfidf_word_cosine'] = np.array(v1w.multiply(v2w).sum(axis=1)).flatten()
    X = feature_df.values.astype(np.float64)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X


print("Feature extraction functions defined.")

## 4. Load Pre-trained Model

The model artefacts include:
- `best_model.pkl`: The trained Gradient Boosting classifier
- `scaler.pkl`: StandardScaler fitted on training data
- `char_vectorizer.pkl`: TF-IDF vectorizer for character n-grams
- `word_vectorizer.pkl`: TF-IDF vectorizer for word n-grams

In [ ]:
MODEL_DIR = 'models'

print(f"Loading model artefacts from: {MODEL_DIR}")

with open(os.path.join(MODEL_DIR, 'best_model.pkl'), 'rb') as f:
    model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'char_vectorizer.pkl'), 'rb') as f:
    char_vectorizer = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'word_vectorizer.pkl'), 'rb') as f:
    word_vectorizer = pickle.load(f)

print(f"Model loaded: {type(model).__name__}")
print("All artefacts loaded successfully.")

## 5. Load Test Data and Generate Predictions

The input file should be a CSV with columns `text_1` and `text_2`.
The `label` column is not required (it won't be present in the test data).

In [ ]:
# ===== CHANGE THIS PATH TO YOUR TEST DATA FILE =====
INPUT_FILE = '../trial_data/AV_trial.csv'
# ====================================================

print(f"Loading data from: {INPUT_FILE}")
test_df = pd.read_csv(INPUT_FILE)
texts1 = test_df['text_1'].fillna('').tolist()
texts2 = test_df['text_2'].fillna('').tolist()
print(f"Loaded {len(texts1)} text pairs.")
test_df.head()

In [ ]:
# Extract features and generate predictions
print("Extracting features...")
X_test = build_feature_matrix(texts1, texts2, char_vectorizer, word_vectorizer)
print(f"Feature matrix shape: {X_test.shape}")

X_test_scaled = scaler.transform(X_test)
predictions = model.predict(X_test_scaled)

print(f"\nPredictions generated: {len(predictions)} total")
print(f"  Predicted 0 (Different Author): {(predictions == 0).sum()}")
print(f"  Predicted 1 (Same Author):      {(predictions == 1).sum()}")

## 6. Save Predictions

In [ ]:
OUTPUT_FILE = 'predictions_A.csv'

pred_df = pd.DataFrame({'prediction': predictions.astype(int)})
pred_df.to_csv(OUTPUT_FILE, index=False)
print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"\nFirst 10 predictions:")
print(pred_df.head(10))

## 7. Optional: Evaluate Against Ground Truth

If ground truth labels are available (e.g., for the trial or dev set),
the predictions are evaluated here.

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix

if 'label' in test_df.columns:
    y_true = test_df['label'].astype(int).values
    
    print("=" * 50)
    print("  EVALUATION RESULTS")
    print("=" * 50)
    print(f"Accuracy:       {accuracy_score(y_true, predictions):.4f}")
    print(f"F1 (macro):     {f1_score(y_true, predictions, average='macro'):.4f}")
    print(f"F1 (weighted):  {f1_score(y_true, predictions, average='weighted'):.4f}")
    print()
    print(classification_report(y_true, predictions, 
                                target_names=['Different Author (0)', 'Same Author (1)']))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, predictions))
else:
    print("No 'label' column found in input file. Skipping evaluation.")